# TSViT para segmentacion temporal de cultivos: el aporte de la fenologia

Este notebook documenta el entrenamiento de **TSViT** (*Temporo-Spatial Vision
Transformer*, Tarasiou et al. 2023), una arquitectura que segmenta cultivos
pixel por pixel consumiendo la **serie temporal completa** de imagenes
Sentinel-2, no una sola fecha.

Lo que distingue a TSViT de una CNN 2D clasica es como organiza la atencion.
Primero aplica un **encoder temporal** sobre la secuencia de `T` fechas de
adquisicion de cada parche, invirtiendo el orden habitual de un video-ViT
(que mira primero el espacio); recien despues aplica el **encoder espacial**.
Usa **K cls tokens** (uno por clase de cultivo) para que cada clase tenga su
propio canal de prediccion, y un **positional encoding temporal indexado por la
fecha real de adquisicion** (el dia del ano, DOY). Esto ultimo es critico:
Sentinel-2 tiene revisitas irregulares porque las nubes obligan a descartar
fechas, asi que el modelo necesita saber *cuando* se tomo cada imagen, no solo
su posicion en la secuencia.

Comparamos dos variantes de la misma arquitectura:

- **Variante base (`tsvit`)**: solo el transformer factorizado temporal-espacial.
- **Variante con fenologia (`tsvit-pheno`)**: la misma red mas una rama
  semantica que alinea por contraste las features visuales de cada pixel con
  descripciones fenologicas por clase generadas por un LLM (metodo de Wen et
  al. 2025, *Phenology Description is All You Need!*). Su Tabla 2 muestra que
  describir la fenologia casi **duplica el F1** en escenario zero-shot.

El notebook entrena ambas variantes y muestra el **delta** lado a lado: cuanto
aporta la fenologia. **No entrena en linea**: dispara la interfaz de linea de
comandos de entrenamiento por subprocess para que cada corrida quede registrada
en MLflow con sus metricas y versiones de codigo y datos, y luego lee y muestra
esos resultados.

## Requisitos para ejecucion end-to-end

- `data/PASTIS-R/` descomprimido (parches Sentinel-2 + mascaras por pixel).
- Dependencias instaladas via `poetry install --with ml`.
- GPU recomendada para el entrenamiento real; en CPU el modo rapido sigue
  siendo ejecutable.

Si el dataset o las dependencias de entrenamiento no estan disponibles, el
notebook continua en modo degradado: muestra el comando que se habria
ejecutado y un mensaje claro, sin romper la ejecucion.

In [ ]:
# Celda de parametros (papermill). El default es BAJO a proposito para que la
# ejecucion end-to-end sea rapida; el entrenamiento real se lanza con, por
# ejemplo, `-p run_full True -p epochs 30`.
epochs = 2
batch_size = 4
n_timesteps = 10
target = "semantic18"
device = "auto"
run_full = False
figures_dir = "paper/figures/us-025"

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
from IPython.display import Markdown, display

# Bootstrap repo root: localiza el pyproject.toml subiendo desde el CWD para
# que el notebook funcione desde notebooks/models/ sin asumir el nombre de
# carpeta.
_REPO_BOOTSTRAP = Path.cwd().resolve()
for _candidate in (_REPO_BOOTSTRAP, *_REPO_BOOTSTRAP.parents):
    if (_candidate / "pyproject.toml").is_file():
        _REPO_BOOTSTRAP = _candidate
        break
if str(_REPO_BOOTSTRAP) not in sys.path:
    sys.path.insert(0, str(_REPO_BOOTSTRAP))

from ml.utils.notebook_setup import find_repo_root
from ml.utils.mlflow_utils import resolve_tracking_uri

# Polars: rendering rico HTML en Jupyter.
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(60)

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 200

%load_ext autoreload
%autoreload 2

REPO = find_repo_root()
FIGURES = REPO / figures_dir
FIGURES.mkdir(parents=True, exist_ok=True)

# Nombres de run MLflow por variante (deben coincidir con los del CLI de
# entrenamiento para poder recuperar las corridas despues).
RUN_NAMES = {
    "tsvit": "alt-tsvit-v1",
    "tsvit-pheno": "alt-tsvit-pheno-v1",
}

display(Markdown(
    f"**Configuracion** — variantes: `tsvit`, `tsvit-pheno` · "
    f"epochs: `{epochs}` · n_timesteps: `{n_timesteps}` · target: `{target}` · "
    f"modo: `{'completo' if run_full else 'rapido'}`"
))

## Seccion 1 — La arquitectura TSViT

TSViT segmenta sobre **PASTIS-R**, un benchmark donde cada muestra es un parche
Sentinel-2 observado varias veces a lo largo del ano, con una mascara que
etiqueta cada pixel con su tipo de cultivo. El modelo recibe la serie temporal
completa del parche (los `n_timesteps` pasos) y produce una clase por pixel.

La clave del diseno es el orden de los dos encoders y como tratan el tiempo:

- **Encoder temporal primero**: a diferencia de un video-ViT, que razona
  primero sobre el espacio y luego sobre el tiempo, TSViT invierte el orden.
  Cada posicion espacial se procesa primero como una secuencia temporal, de
  modo que el modelo aprende la *trayectoria* de cada pixel a lo largo de la
  temporada antes de mirar a sus vecinos.
- **Encoder espacial despues**: una vez resumida la dinamica temporal, una
  segunda etapa de atencion integra el contexto espacial dentro del parche
  (bordes de parcela, vecindad, textura).
- **K cls tokens**: la arquitectura introduce un token de clase por cada una de
  las `K` clases de cultivo. Cada token agrega evidencia para su clase, lo que
  da a la red un canal dedicado por cultivo en lugar de un unico vector
  compartido.
- **Positional encoding temporal por DOY real**: el encoding posicional del eje
  temporal no usa el indice de la secuencia (1, 2, 3, ...) sino el **dia del
  ano** real de cada adquisicion. Esto es critico porque Sentinel-2 sufre
  revisitas irregulares: las nubes obligan a descartar fechas, asi que la
  separacion entre dos observaciones consecutivas no es uniforme. Indexar por
  DOY le dice al modelo que tan lejos en el calendario estan dos imagenes.

Las dos variantes comparten exactamente este nucleo; solo difieren en si se
agrega la rama fenologica contrastiva (Seccion 3).

In [ ]:
def run_training(model_kind: str, n_epochs: int) -> dict[str, float | str | None]:
    """Lanza el CLI de entrenamiento de una variante TSViT y parsea sus metricas.

    Invoca `python -m ml.train.train_segmentation` por subprocess para que la
    corrida quede registrada en MLflow. El notebook documenta la invocacion
    CLI: por eso subprocess es la forma correcta aqui, no importar y llamar la
    funcion en linea. Sirve para ambas variantes (`tsvit` y `tsvit-pheno`)
    porque el comando solo cambia en el flag `--model`.

    Args:
        model_kind: 'tsvit' o 'tsvit-pheno'.
        n_epochs: Numero de epochs a entrenar.

    Returns:
        Diccionario con `model`, `miou`, `f1_macro`, `pixel_acc`, `returncode`
        y `error`. Las metricas son `None` si la corrida fallo o no se pudo
        parsear (modo degradado).
    """
    cmd = [
        sys.executable, "-m", "ml.train.train_segmentation",
        "--model", model_kind,
        "--epochs", str(n_epochs),
        "--batch-size", str(batch_size),
        "--n-timesteps", str(n_timesteps),
        "--target", target,
        "--device", device,
        "--run-name", RUN_NAMES.get(model_kind, model_kind),
    ]
    display(Markdown(f"`{' '.join(cmd)}`"))

    result: dict[str, float | str | None] = {
        "model": model_kind,
        "miou": None,
        "f1_macro": None,
        "pixel_acc": None,
        "returncode": None,
        "error": None,
    }
    try:
        proc = subprocess.run(
            cmd, cwd=str(REPO), capture_output=True, text=True, check=False
        )
    except OSError as exc:
        result["error"] = f"No se pudo lanzar el subprocess: {exc}"
        print(f"  {model_kind}: subprocess no disponible ({exc})")
        return result

    result["returncode"] = proc.returncode
    log = (proc.stdout or "") + "\n" + (proc.stderr or "")

    if proc.returncode != 0:
        # Modo degradado: se muestra el final del log pero no se rompe el
        # notebook (el dataset o la GPU pueden no estar disponibles en CI).
        tail = "\n".join(log.strip().splitlines()[-12:])
        result["error"] = f"returncode={proc.returncode}"
        print(f"  {model_kind}: entrenamiento fallido (returncode={proc.returncode})")
        if tail:
            display(Markdown(f"```\n{tail}\n```"))
        return result

    # El CLI loguea `cli_done` (structlog) con las metricas del mejor epoch.
    # structlog renderiza key=value; parseamos miou/f1_macro/pixel_acc.
    for line in reversed(log.splitlines()):
        if "cli_done" in line:
            for key in ("miou", "f1_macro", "pixel_acc"):
                token = f"{key}="
                if token in line:
                    raw = line.split(token, 1)[1].split()[0].rstrip(",")
                    try:
                        result[key] = float(raw)
                    except ValueError:
                        result[key] = None
            break

    miou = result["miou"]
    if isinstance(miou, float):
        print(f"  {model_kind}: mIoU={miou:.4f}")
    else:
        print(f"  {model_kind}: corrida OK pero no se parsearon metricas del log")
    return result

## Seccion 2 — Variante base (sin fenologia)

La primera corrida entrena TSViT en su forma pura: solo el transformer
factorizado temporal-espacial descrito arriba, sin ninguna ayuda semantica
externa. Aprende a separar cultivos unicamente a partir de la evolucion
espectral de cada pixel a lo largo de la temporada y del contexto espacial del
parche. Esta variante es la **linea base** contra la que mediremos el aporte de
la fenologia.

El CLI registra la corrida en el experimento `agrosat-segmentation` con el
nombre `alt-tsvit-v1` y los tags `code_version` (SHA git) y `data_version`
(hash DVC del dataset).

In [ ]:
# Variante base: TSViT sin fenologia. En modo rapido se usa el `epochs` bajo de
# la celda de parametros; en modo completo (run_full) tambien, pero se espera
# que el valor venga elevado via papermill (`-p epochs 30`).
n_epochs = epochs

display(Markdown("### Entrenando `tsvit` (base)"))
result_base = run_training("tsvit", n_epochs)

base_df = pl.DataFrame(
    [result_base],
    schema={
        "model": pl.Utf8,
        "miou": pl.Float64,
        "f1_macro": pl.Float64,
        "pixel_acc": pl.Float64,
        "returncode": pl.Int64,
        "error": pl.Utf8,
    },
)
display(base_df.select("model", "miou", "f1_macro", "pixel_acc", "returncode"))

## Seccion 3 — Variante con fenologia contrastiva

La segunda corrida agrega a TSViT la **rama fenologica-contrastiva** propuesta
por Wen et al. (2025). La idea es ensenarle al encoder visual *que patron
temporal corresponde a cada cultivo* usando texto como guia.

El procedimiento tiene tres pasos:

1. **Descripcion fenologica por clase**: para cada cultivo se resume su curva
   NDVI media a lo largo de la temporada (cuando germina, cuando alcanza su
   maximo de verdor, cuando se cosecha) y se entrega esa descripcion a un LLM,
   que genera un texto fenologico de la clase.
2. **Embedding del texto**: ese texto se convierte en un vector (prototipo
   semantico de la clase).
3. **Alineamiento por contraste (InfoNCE)**: durante el entrenamiento, las
   features visuales de cada pixel se acercan al prototipo de su clase y se
   alejan de los prototipos de las demas clases. Asi el texto fenologico actua
   como ancla: empuja las features de cada pixel hacia el cluster semantico de
   su cultivo.

El nucleo TSViT es identico al de la Seccion 2; lo unico que cambia es este
termino contrastivo adicional. La corrida se registra como `alt-tsvit-pheno-v1`.

In [ ]:
display(Markdown("### Entrenando `tsvit-pheno` (con fenologia)"))
result_pheno = run_training("tsvit-pheno", n_epochs)

pheno_df = pl.DataFrame(
    [result_pheno],
    schema={
        "model": pl.Utf8,
        "miou": pl.Float64,
        "f1_macro": pl.Float64,
        "pixel_acc": pl.Float64,
        "returncode": pl.Int64,
        "error": pl.Utf8,
    },
)
display(pheno_df.select("model", "miou", "f1_macro", "pixel_acc", "returncode"))

## Seccion 4 — Comparativa: el aporte de la fenologia

Con ambas corridas listas, ponemos las dos variantes lado a lado y calculamos
el **delta** de cada metrica (`tsvit-pheno` menos `tsvit`). Un delta positivo
significa que describir la fenologia ayudo a segmentar mejor.

Wen et al. (2025) reportan que el alineamiento fenologico casi duplica el F1 en
el escenario zero-shot (su Tabla 2). Aqui no estamos en zero-shot, asi que la
mejora esperada es mas moderada, pero la direccion deberia ser la misma.

In [ ]:
# Combina las dos variantes y calcula el delta de cada metrica.
results_df = pl.concat([base_df, pheno_df])
by_model = {row["model"]: row for row in results_df.iter_rows(named=True)}
base = by_model.get("tsvit")
pheno = by_model.get("tsvit-pheno")

delta_df: pl.DataFrame | None = None
if base is None or pheno is None:
    display(Markdown(
        "> Se necesitan ambas corridas `tsvit` y `tsvit-pheno`. Modo degradado."
    ))
elif base["miou"] is None or pheno["miou"] is None:
    display(Markdown(
        "> Una de las corridas (`tsvit` / `tsvit-pheno`) no produjo metricas; "
        "no se puede calcular el delta. Modo degradado."
    ))
else:
    delta_rows = []
    for metric in ("miou", "f1_macro", "pixel_acc"):
        b = float(base[metric])
        p = float(pheno[metric])
        delta_rows.append({
            "metrica": metric,
            "tsvit": round(b, 4),
            "tsvit_pheno": round(p, 4),
            "delta": round(p - b, 4),
        })
    delta_df = pl.DataFrame(delta_rows)
    display(delta_df)

    miou_delta = delta_df.filter(pl.col("metrica") == "miou")["delta"][0]
    verdict = "ayuda" if miou_delta > 0 else ("no ayuda" if miou_delta < 0 else "no cambia")
    display(Markdown(
        f"La rama fenologica **{verdict}**: el mIoU varia en `{miou_delta:+.4f}` "
        "respecto al TSViT sin fenologia."
    ))

In [ ]:
# Grafico de barras comparando ambas variantes en mIoU y F1-macro. Si una
# corrida no produjo metricas, su barra queda en 0.
plot_df = results_df.with_columns(
    pl.col("miou").fill_null(0.0),
    pl.col("f1_macro").fill_null(0.0),
)

models = plot_df["model"].to_list()
miou_vals = plot_df["miou"].to_list()
f1_vals = plot_df["f1_macro"].to_list()

if models:
    x = range(len(models))
    width = 0.38
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar([i - width / 2 for i in x], miou_vals, width, label="mIoU")
    ax.bar([i + width / 2 for i in x], f1_vals, width, label="F1-macro")
    ax.set_xticks(list(x))
    ax.set_xticklabels(models)
    ax.set_ylabel("Metrica de validacion")
    ax.set_title("TSViT con y sin fenologia — mejor epoch")
    ax.set_ylim(0.0, 1.0)
    ax.legend()
    fig.tight_layout()

    out_path = FIGURES / "tsvit_pheno_comparison.png"
    fig.savefig(out_path, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    display(Markdown(f"Figura guardada en `{out_path.relative_to(REPO)}`."))
else:
    display(Markdown("> Sin variantes para graficar."))

In [ ]:
# Lectura de las corridas registradas en MLflow para verificar el lineage.
# resolve_tracking_uri() elige el servidor MLflow local si responde, o el file
# store ./mlruns como fallback.
EXPERIMENT_NAME = "agrosat-segmentation"
TSVIT_RUN_NAMES = {"alt-tsvit-v1", "alt-tsvit-pheno-v1"}

mlflow_runs_df: pl.DataFrame | None = None
try:
    import mlflow

    mlflow.set_tracking_uri(resolve_tracking_uri())
    exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if exp is None:
        display(Markdown(
            f"> El experimento `{EXPERIMENT_NAME}` aun no existe en MLflow "
            "(ninguna corrida se registro). Modo degradado."
        ))
    else:
        runs_pd = mlflow.search_runs(
            experiment_ids=[exp.experiment_id],
            order_by=["attributes.start_time DESC"],
            max_results=50,
        )
        if runs_pd.empty or "tags.mlflow.runName" not in runs_pd.columns:
            display(Markdown("> No hay corridas registradas todavia. Modo degradado."))
        else:
            runs_pd = runs_pd[runs_pd["tags.mlflow.runName"].isin(TSVIT_RUN_NAMES)]
            if runs_pd.empty:
                display(Markdown(
                    "> No se encontraron corridas `alt-tsvit-v1` / "
                    "`alt-tsvit-pheno-v1`. Modo degradado."
                ))
            else:
                keep = [
                    c for c in (
                        "tags.mlflow.runName",
                        "metrics.best_val_miou",
                        "metrics.best_val_f1_macro",
                        "metrics.best_val_pixel_acc",
                        "tags.code_version",
                        "tags.data_version",
                    )
                    if c in runs_pd.columns
                ]
                mlflow_runs_df = pl.from_pandas(runs_pd[keep]).rename({
                    "tags.mlflow.runName": "run_name",
                    "metrics.best_val_miou": "miou",
                    "metrics.best_val_f1_macro": "f1_macro",
                    "metrics.best_val_pixel_acc": "pixel_acc",
                    "tags.code_version": "code_version",
                    "tags.data_version": "data_version",
                })
                display(mlflow_runs_df)
except Exception as exc:  # noqa: BLE001 - modo degradado en notebook
    display(Markdown(f"> MLflow no disponible para lectura: `{exc}`. Modo degradado."))

## Conclusiones

Entrenamos **TSViT** en dos variantes para segmentar cultivos pixel por pixel
sobre parches satelitales del sur de Francia. TSViT lee la serie temporal
completa de cada parche (varias fechas Sentinel-2) en lugar de una sola imagen,
razonando primero sobre como cambia cada pixel a lo largo de la temporada y
despues sobre su entorno espacial. Ambas variantes se evaluaron con la misma
vara: **mIoU** (cuanto se solapan la prediccion y la verdad para cada clase,
promediado), **F1-macro** (equilibrio entre aciertos y falsos positivos,
tratando todas las clases por igual) y **exactitud por pixel** (fraccion de
pixeles bien clasificados).

Los hallazgos concretos salen de las tablas de este notebook:

- La variante base (`tsvit`) fija el punto de partida con su mIoU y F1-macro de
  la Seccion 2: lo que logra el transformer temporal-espacial por si solo.
- La variante con fenologia (`tsvit-pheno`) anade una rama que alinea las
  features de cada pixel con una descripcion fenologica de su clase. El
  **delta** de la Seccion 4 dice si esa ayuda semantica mejoro la segmentacion
  y por cuanto.
- Si el delta de mIoU es positivo, la fenologia funciono: describir *como* luce
  cada cultivo a lo largo del calendario y empujar las features hacia ese
  prototipo separa clases que de otra forma se confunden, en linea con la
  hipotesis de Wen et al. (2025). Si es nulo o negativo, con estas condiciones
  (pocos epochs, sin zero-shot) la senal de texto aun no compensa el termino
  contrastivo adicional.

Conviene recordar que estos numeros salen del modo rapido (pocos epochs),
pensado para verificar que el flujo completo corre de principio a fin. No son
todavia las metricas finales: con tan pocas pasadas la red apenas empieza a
aprender, y el efecto de la fenologia suele notarse mas cuando el entrenamiento
es largo.

### Lo que sigue

- Lanzar el entrenamiento completo en una GPU L4/H100 con mas epochs
  (`-p run_full True -p epochs 30`) para obtener las metricas definitivas de
  ambas variantes y un delta confiable.
- Si la rama fenologica confirma la mejora, refinar las descripciones por clase
  y ajustar el peso del termino contrastivo.
- Integrar la mejor variante de TSViT al ensemble final, junto con los demas
  segmentadores densos, para decidir la combinacion ganadora.